### **[Machine Learning Models in Crypto Trading](https://medium.com/@laostjen/78a6735b5639)**

In [ ]:
from sklearn.preprocessing import StandardScaler
# RIGHT: Fit scaler only on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
# Use training statistics to transform test data
X_test_scaled = scaler.transform(X_test)

In [ ]:
# RIGHT: Only use past volatility
df['past_volatility'] = df['returns'].rolling(24).std()

# Engineered features that encode market structure
features = [
  'returns', 'log_returns', 'volatility',
  'volume_change', 'volume_momentum',
  'price_momentum_1h', 'price_momentum_4h', 'price_momentum_24h',
  'rsi', 'macd', 'bollinger_position',
  'order_book_imbalance', 'trade_flow_imbalance',
  'realized_volatility', 'garman_klass_volatility',
  # ... market microstructure features
]

In [ ]:
model = Sequential([
  LSTM(128, return_sequences=True),
  Dropout(0.3),
  LSTM(64, return_sequences=True),
  Dropout(0.3),
  LSTM(32),
  Dense(16, activation='relu'),
  Dense(1, activation='sigmoid')
])

# Training on 2 years of hourly data (17,520 samples)
# Model had 87,000+ parameters

# Result: 91% training accuracy, 52% test accuracy (no better than random)

In [ ]:
# Testing stationarity of Bitcoin returns
from statsmodels.tsa.stattools import adfuller

def test_stationarity(timeseries):
  result = adfuller(timeseries)
  print(f'ADF Statistic: {result[0]}')
  print(f'p-value: {result[1]}')
  return result[1] < 0.05  # True if stationary

# Testing different periods
bull_market = btc_returns['2020-01':'2021-11']
bear_market = btc_returns['2022-01':'2022-12']

print("Bull market stationary:", test_stationarity(bull_market))
# Output: False (p-value: 0.23)

print("Bear market stationary:", test_stationarity(bear_market))
# Output: False (p-value: 0.19)

In [ ]:
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

class CryptoDirectionModel:
  """
  LightGBM model for predicting short-term price direction
  """
  def __init__(self):
    self.model = None
    self.feature_names = None

  def create_features(self, df):
    """
    Create features from OHLCV data
    """
    features = pd.DataFrame(index=df.index)

    # Price-based features
    features['returns'] = df['close'].pct_change()
    features['log_returns'] = np.log(df['close'] / df['close'].shift(1))

    # Momentum features (multiple timeframes)
    for period in [3, 6, 12, 24]:
      features[f'momentum_{period}h'] = (
        df['close'] / df['close'].shift(period) - 1
      )

    # Volatility features
    features['volatility_6h'] = features['returns'].rolling(6).std()
    features['volatility_24h'] = features['returns'].rolling(24).std()

    # Volume features
    features['volume_change'] = df['volume'].pct_change()
    features['volume_momentum_6h'] = (
      df['volume'] / df['volume'].rolling(6).mean()
    )

    # Technical indicators
    features['rsi_14'] = self.calculate_rsi(df['close'], 14)
    features['macd'], features['macd_signal'] = self.calculate_macd(df['close'])

    # Order flow features (if available)
    if 'bid_volume' in df.columns:
      features['order_imbalance'] = (
        (df['bid_volume'] - df['ask_volume']) /
        (df['bid_volume'] + df['ask_volume'])
      )

    return features.dropna()

  def create_labels(self, df, forward_hours=4, threshold=0.005):
    """
    Create labels: 1 if price rises >0.5% in next 4 hours, 0 otherwise
    """
    future_return = (
      df['close'].shift(-forward_hours) / df['close'] - 1
    )
    labels = (future_return > threshold).astype(int)
    return labels

  def train(self, df, validation_split=0.2):
    """
    Train model with time-aware validation
    """
    # Create features and labels
    X = self.create_features(df)
    y = self.create_labels(df)

    # Align features and labels
    valid_idx = X.index.intersection(y.index)
    X = X.loc[valid_idx]
    y = y.loc[valid_idx]

    # Time-based split (no shuffling - preserves temporal order)
    split_point = int(len(X) * (1 - validation_split))
    X_train, X_val = X[:split_point], X[split_point:]
    y_train, y_val = y[:split_point], y[split_point:]

    # Model parameters (conservative to prevent overfitting)
    params = {
      'objective': 'binary',
      'metric': 'auc',
      'num_leaves': 31,
      'learning_rate': 0.05,
      'feature_fraction': 0.8,
      'bagging_fraction': 0.8,
      'bagging_freq': 5,
      'max_depth': 6,
      'min_data_in_leaf': 100,
      'verbose': -1
    }

    # Train with early stopping
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

    self.model = lgb.train(
      params,
      train_data,
      num_boost_round=1000,
      valid_sets=[train_data, val_data],
      callbacks=[lgb.early_stopping(stopping_rounds=50)]
    )

    self.feature_names = X.columns.tolist()

    # Evaluation
    y_pred_val = self.model.predict(X_val)
    auc_score = roc_auc_score(y_val, y_pred_val)

    print(f"Validation AUC: {auc_score:.4f}")

    return self.model

  def predict_probability(self, df):
    """
    Predict probability of upward movement
    """
    X = self.create_features(df)
    probabilities = self.model.predict(X)
    return probabilities

  def get_feature_importance(self, top_n=15):
    """
    Get most important features
    """
    importance = self.model.feature_importance(importance_type='gain')
    feature_importance = pd.DataFrame({
      'feature': self.feature_names,
      'importance': importance
    }).sort_values('importance', ascending=False)

    return feature_importance.head(top_n)

  @staticmethod
  def calculate_rsi(prices, period=14):
    """Calculate RSI indicator"""
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

  @staticmethod
  def calculate_macd(prices, fast=12, slow=26, signal=9):
    """Calculate MACD indicator"""
    ema_fast = prices.ewm(span=fast).mean()
    ema_slow = prices.ewm(span=slow).mean()
    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=signal).mean()
    return macd, macd_signal

# Usage example
model = CryptoDirectionModel()
model.train(historical_data)

# Get predictions
current_prob = model.predict_probability(current_data)
print(f"Probability of 0.5%+ rise in next 4h: {current_prob[-1]:.2%}")

# Understand what drives predictions
print("\nTop features:")
print(model.get_feature_importance())

In [ ]:
class EnsemblePredictor:
  """
  Ensemble of specialized models for robust predictions
  """
  def __init__(self):
    self.models = {
      'direction_4h': CryptoDirectionModel(),  # Short-term direction
      'volatility': VolatilityRegressor(),      # Volatility forecast
      'regime': RegimeClassifier()              # Market regime
    }
    self.meta_model = None

  def train_base_models(self, train_data):
    """Train all base models"""
    for name, model in self.models.items():
      print(f"Training {name}...")
      model.train(train_data)

  def train_meta_model(self, val_data):
    """
    Train meta-model that learns how to combine base model predictions
    """
    # Get predictions from all base models
    base_predictions = {}
    for name, model in self.models.items():
      base_predictions[name] = model.predict(val_data)

    # Create meta-features
    X_meta = pd.DataFrame(base_predictions)
    y_meta = self.create_labels(val_data)

    # Train simple logistic regression as meta-model
    from sklearn.linear_model import LogisticRegression
    self.meta_model = LogisticRegression()
    self.meta_model.fit(X_meta, y_meta)

  def predict(self, current_data):
    """Final ensemble prediction"""
    base_predictions = {}
    for name, model in self.models.items():
      base_predictions[name] = model.predict(current_data)

    X_meta = pd.DataFrame([base_predictions])
    final_prediction = self.meta_model.predict_proba(X_meta)[0, 1]

    return final_prediction

In [ ]:
from river import linear_model, preprocessing, compose, metrics

class OnlineAdaptiveModel:
  """
  Online learning model that adapts to market changes
  """
  def __init__(self):
    # Create online learning pipeline
    self.model = compose.Pipeline(
      preprocessing.StandardScaler(),
      linear_model.LogisticRegression()
    )
    self.metric = metrics.ROCAUC()

  def update_and_predict(self, features, label=None):
    """
    Predict, then update model with new data
    """
    # Make prediction first
    y_pred = self.model.predict_proba_one(features)

    # Then update with true label (if available)
    if label is not None:
      self.model.learn_one(features, label)
      self.metric.update(label, y_pred.get(True, 0))

    return y_pred.get(True, 0)  # Return probability of positive class

  def get_current_performance(self):
    """Get rolling performance metric"""
    return self.metric.get()

# Usage in production
online_model = OnlineAdaptiveModel()

for timestamp, data in live_data_stream:
  features = extract_features(data)

  # Predict
  prediction = online_model.update_and_predict(features)

  # Wait for outcome, then update
  actual_outcome = get_actual_outcome(timestamp, hours_ahead=4)
  online_model.update_and_predict(features, actual_outcome)

  # Monitor performance
  if timestamp % 100 == 0:
    print(f"Current AUC: {online_model.get_current_performance():.4f}")

In [ ]:
import gym
import numpy as np
from stable_baselines3 import PPO

class PositionSizingEnv(gym.Env):
  """
  RL environment for learning optimal position sizing
  """
  def __init__(self, data, base_signals):
    super().__init__()
    self.data = data
    self.base_signals = base_signals  # From our direction model
    self.current_step = 0
    self.position = 0
    self.cash = 100000

    # Action space: position size (0 to 1.0 of capital)
    self.action_space = gym.spaces.Box(
      low=0, high=1, shape=(1,), dtype=np.float32
    )

    # Observation space: market features + current position
    self.observation_space = gym.spaces.Box(
      low=-np.inf, high=np.inf, shape=(10,), dtype=np.float32
    )

  def reset(self):
    self.current_step = 0
    self.position = 0
    self.cash = 100000
    return self._get_observation()

  def step(self, action):
    # Action is desired position size
    desired_position_size = action[0]

    # Get current signal and market state
    current_signal = self.base_signals[self.current_step]
    current_price = self.data['close'].iloc[self.current_step]
    next_price = self.data['close'].iloc[self.current_step + 1]

    # Calculate return
    if current_signal > 0.6:  # Bullish signal
      # RL determines how much to bet
      actual_position = desired_position_size * self.cash / current_price
      pnl = actual_position * (next_price - current_price)
    else:
      pnl = 0  # No trade

    self.cash += pnl
    self.current_step += 1

    # Reward: Sharpe-like (return / volatility)
    reward = pnl / (self.cash * 0.02)  # Normalized by portfolio and volatility

    done = self.current_step >= len(self.data) - 1

    return self._get_observation(), reward, done, {}

  def _get_observation(self):
      # Current state: market features + position
      idx = self.current_step
      obs = np.array([
          self.data['returns'].iloc[idx],
          self.data['volatility_24h'].iloc[idx],
          self.base_signals[idx],
          self.position / self.cash if self.cash > 0 else 0,
          # ... additional features
      ], dtype=np.float32)
      return obs

# Training (offline, on historical data)
env = PositionSizingEnv(train_data, train_signals)
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=100000)

# Using in production
def determine_position_size(market_state, base_signal):
  obs = create_observation(market_state, base_signal)
  position_size = model.predict(obs, deterministic=True)[0]
  return position_size

In [ ]:
def create_momentum_features(df):
  """Momentum across multiple timeframes captures trend strength"""
  features = pd.DataFrame(index=df.index)

  for hours in [1, 4, 12, 24, 72, 168]:  # 1h to 1 week
    features[f'momentum_{hours}h'] = (
      df['close'] / df['close'].shift(hours) - 1
    )

    # Also capture acceleration (momentum of momentum)
    features[f'momentum_acceleration_{hours}h'] = (
      features[f'momentum_{hours}h'].diff()
    )

  return features


def create_volatility_features(df):
  """Multiple volatility estimators for robustness"""
  features = pd.DataFrame(index=df.index)

  # Simple realized volatility
  returns = df['close'].pct_change()
  features['rv_6h'] = returns.rolling(6).std() * np.sqrt(365*24)  # Annualized
  features['rv_24h'] = returns.rolling(24).std() * np.sqrt(365*24)

  # Parkinson (uses high/low, more efficient)
  features['parkinson_24h'] = np.sqrt(
    1/(4*np.log(2)) *
    (np.log(df['high']/df['low'])**2).rolling(24).mean()
  ) * np.sqrt(365*24)

  # Garman-Klass (uses OHLC, even more efficient)
  hl = np.log(df['high']/df['low'])**2
  co = np.log(df['close']/df['open'])**2
  features['gk_24h'] = np.sqrt(
    0.5*hl.rolling(24).mean() - (2*np.log(2)-1)*co.rolling(24).mean()
  ) * np.sqrt(365*24)

  # Volatility changes (regime shifts)
  features['vol_change_6h'] = features['rv_6h'] / features['rv_24h']

  return features


def create_orderflow_features(df):
  """
  Order book and trade flow features (exchange API required)
  """
  features = pd.DataFrame(index=df.index)

  # Order book imbalance (bid vs ask pressure)
  total_liquidity = df['bid_volume_sum'] + df['ask_volume_sum']
  features['ob_imbalance'] = (
    (df['bid_volume_sum'] - df['ask_volume_sum']) / total_liquidity
  )

  # Trade flow (buy vs sell market orders)
  total_trades = df['buy_volume'] + df['sell_volume']
  features['trade_imbalance'] = (
    (df['buy_volume'] - df['sell_volume']) / total_trades
  )

  # Imbalance momentum
  features['imbalance_momentum'] = (
    features['trade_imbalance'].diff().rolling(6).mean()
  )

  return features


def create_risk_adjusted_features(df):
  """Returns normalized by volatility for comparability"""
  features = pd.DataFrame(index=df.index)

  returns = df['close'].pct_change()
  volatility = returns.rolling(24).std()

  # Sharpe-like: return normalized by volatility
  features['risk_adj_return_6h'] = (
    returns.rolling(6).mean() / (volatility + 1e-8)
  )

  # Sortino-like: return normalized by downside volatility
  downside_vol = returns[returns < 0].rolling(24).std()
  features['sortino_return_6h'] = (
    returns.rolling(6).mean() / (downside_vol + 1e-8)
  )

  return features


def create_microstructure_features(df):
  """Price impact and liquidity measures"""
  features = pd.DataFrame(index=df.index)

  # Amihud illiquidity: price impact per dollar traded
  features['illiquidity'] = (
    abs(df['close'].pct_change()) / (df['volume'] * df['close'] + 1e-8)
  ).rolling(24).mean()

  # Bid-ask spread (when available)
  if 'bid' in df.columns and 'ask' in df.columns:
    features['spread'] = (df['ask'] - df['bid']) / df['close']
    features['spread_ma'] = features['spread'].rolling(24).mean()

  # Effective spread from trades
  features['effective_spread'] = (
    2 * abs(df['close'] - (df['high'] + df['low'])/2)
  ) / df['close']

  return features

In [ ]:
def evaluate_trading_model(y_true, y_pred_proba, trading_threshold=0.6):
  """
  Evaluate model at actual trading threshold
  """
  # We only trade when probability > 60% or < 40%
  trade_mask = (y_pred_proba > trading_threshold) | (y_pred_proba < (1-trading_threshold))

  # Filter to actual trades
  y_true_trades = y_true[trade_mask]
  y_pred_trades = (y_pred_proba[trade_mask] > 0.5).astype(int)

  from sklearn.metrics import precision_score, recall_score, f1_score

  precision = precision_score(y_true_trades, y_pred_trades)
  recall = recall_score(y_true_trades, y_pred_trades)
  f1 = f1_score(y_true_trades, y_pred_trades)

  print(f"At threshold {trading_threshold}:")
  print(f"  Precision: {precision:.3f}")
  print(f"  Recall: {recall:.3f}")
  print(f"  F1 Score: {f1:.3f}")
  print(f"  Trade frequency: {trade_mask.sum() / len(trade_mask):.1%}")

  return precision, recall, f1


In [ ]:
def calculate_strategy_pnl(df, predictions, threshold=0.6, cost_per_trade=0.002):
  """
  Calculate actual P&L including transaction costs
  """
  positions = np.zeros(len(df))

  # Take position when confident
  positions[predictions > threshold] = 1      # Long
  positions[predictions < (1-threshold)] = -1  # Short (if allowed)

  # Calculate returns
  returns = df['close'].pct_change()
  strategy_returns = positions.shift(1) * returns  # Shift to avoid look-ahead

  # Subtract transaction costs
  position_changes = positions.diff().abs()
  costs = position_changes * cost_per_trade
  strategy_returns_after_costs = strategy_returns - costs

  # Calculate metrics
  cumulative_return = (1 + strategy_returns_after_costs).prod() - 1
  sharpe = strategy_returns_after_costs.mean() / strategy_returns_after_costs.std() * np.sqrt(365*24)
  max_dd = (strategy_returns_after_costs.cumsum().cummax() - strategy_returns_after_costs.cumsum()).max()

  print(f"Cumulative Return: {cumulative_return:.2%}")
  print(f"Sharpe Ratio: {sharpe:.2f}")
  print(f"Max Drawdown: {max_dd:.2%}")
  print(f"Number of trades: {position_changes.sum()}")

  return strategy_returns_after_costs

In [ ]:
def evaluate_by_regime(df, predictions, y_true):
  """
  Check if model works in different market conditions
  """
  # Define regimes
  volatility = df['close'].pct_change().rolling(24).std()

  low_vol = volatility < volatility.quantile(0.33)
  high_vol = volatility > volatility.quantile(0.66)

  from sklearn.metrics import roc_auc_score

  print("Low volatility regime:")
  print(f"  AUC: {roc_auc_score(y_true[low_vol], predictions[low_vol]):.3f}")

  print("Medium volatility regime:")
  mid_vol = ~low_vol & ~high_vol
  print(f"  AUC: {roc_auc_score(y_true[mid_vol], predictions[mid_vol]):.3f}")

  print("High volatility regime:")
  print(f"  AUC: {roc_auc_score(y_true[high_vol], predictions[high_vol]):.3f}")

In [ ]:
# Slow: Computing from scratch every time
def slow_feature_computation(current_data, historical_data):
  """This recomputes everything - too slow"""
  features = {}
  features['momentum_24h'] = (
    current_data['close'] / historical_data['close'].iloc[-24] - 1
  )
  features['volatility_24h'] = (
    historical_data['close'].pct_change().tail(24).std()
  )
  # ... 50 more features
  return features  # Takes 200ms - too slow for HFT

# Fast: Incremental updates
class IncrementalFeatureComputer:
  """Maintains state and updates incrementally"""
  def __init__(self):
    self.price_buffer = deque(maxlen=168)  # 1 week of hourly data
    self.return_buffer = deque(maxlen=168)
    self.rolling_stats = {}

  def update(self, new_candle):
    """Update with new data point (< 1ms)"""
    self.price_buffer.append(new_candle['close'])

    if len(self.price_buffer) > 1:
      ret = new_candle['close'] / self.price_buffer[-2] - 1
      self.return_buffer.append(ret)

    # Incremental statistics
    if len(self.return_buffer) >= 24:
      self.rolling_stats['volatility_24h'] = np.std(
        list(self.return_buffer)[-24:]
      )

    if len(self.price_buffer) >= 24:
      self.rolling_stats['momentum_24h'] = (
        self.price_buffer[-1] / self.price_buffer[-24] - 1
      )

  def get_features(self):
    """Retrieve pre-computed features (instant)"""
    return self.rolling_stats

# In production
feature_computer = IncrementalFeatureComputer()

for candle in live_stream:
  feature_computer.update(candle)  # <1ms
  features = feature_computer.get_features()  # Instant
  prediction = model.predict(features)  # ~5ms
  # Total: ~6ms (acceptable)

In [ ]:
class ModelManager:
  """Manages multiple model versions in production"""
  def __init__(self):
    self.models = {
      'production': load_model('model_v12_production.pkl'),
      'shadow': load_model('model_v13_shadow.pkl')
    }
    self.allocation = {'production': 1.0, 'shadow': 0.0}  # Shadow gets no capital

  def predict(self, features, account_type='production'):
    """Get prediction from specified model"""
    model = self.models[account_type]
    prediction = model.predict(features)

    # Log for comparison
    log_prediction(account_type, prediction, features)

    return prediction

  def compare_performance(self, days=7):
    """Compare production vs shadow model"""
    prod_performance = get_performance('production', days)
    shadow_performance = get_performance('shadow', days)

    print(f"Production (v12) Sharpe: {prod_performance['sharpe']:.2f}")
    print(f"Shadow (v13) Sharpe: {shadow_performance['sharpe']:.2f}")

    if shadow_performance['sharpe'] > prod_performance['sharpe'] * 1.1:
      print("Shadow model outperforming! Consider gradual rollout.")

  def gradual_rollout(self, new_allocation={'production': 0.8, 'shadow': 0.2}):
    """Allocate capital to new model gradually"""
    self.allocation = new_allocation
    print(f"New allocation: {new_allocation}")

# Usage
model_mgr = ModelManager()

for data in live_stream:
  features = extract_features(data)

  # Production model (gets actual capital)
  pred_prod = model_mgr.predict(features, 'production')

  # Shadow model (tracked but no capital risked)
  pred_shadow = model_mgr.predict(features, 'shadow')

  # Execute based on production model only
  execute_trade_if_confident(pred_prod)

# After 1 week of shadow testing
model_mgr.compare_performance(days=7)
# If shadow model better, gradually roll out
model_mgr.gradual_rollout({'production': 0.7, 'shadow': 0.3})

In [ ]:
class ModelMonitor:
  """Monitor model performance and data quality in production"""
  def __init__(self):
    self.prediction_history = []
    self.feature_history = []
    self.outcome_history = []

  def log_prediction(self, features, prediction, actual_outcome=None):
    """Log every prediction for monitoring"""
    self.feature_history.append(features)
    self.prediction_history.append(prediction)
    if actual_outcome is not None:
      self.outcome_history.append(actual_outcome)

  def check_feature_drift(self, window=1000):
    """Detect if feature distributions have changed"""
    recent_features = pd.DataFrame(self.feature_history[-window:])
    baseline_features = pd.DataFrame(self.feature_history[-5000:-window])

    from scipy.stats import ks_2samp

    drift_detected = []
    for col in recent_features.columns:
      stat, pvalue = ks_2samp(
        baseline_features[col].dropna(),
        recent_features[col].dropna()
      )
      if pvalue < 0.01:  # Significant drift
        drift_detected.append(col)

    if drift_detected:
      alert(f"Feature drift detected: {drift_detected}")

    return drift_detected

  def check_prediction_calibration(self, window=1000):
    """Check if predicted probabilities match actual frequencies"""
    recent_preds = self.prediction_history[-window:]
    recent_outcomes = self.outcome_history[-window:]

    # Bin predictions
    bins = [0, 0.4, 0.5, 0.6, 1.0]
    pred_bins = pd.cut(recent_preds, bins)

    # Calculate actual rate in each bin
    df = pd.DataFrame({
      'pred_bin': pred_bins,
      'outcome': recent_outcomes
    })

    calibration = df.groupby('pred_bin')['outcome'].mean()

    print("Calibration check:")
    print(calibration)

    # If poorly calibrated, alert
    # e.g., if predictions of 60% are only correct 52% of the time

  def check_performance_degradation(self, window=500):
    """Check if rolling performance is declining"""
    from sklearn.metrics import roc_auc_score

    # Calculate rolling AUC
    rolling_aucs = []
    for i in range(window, len(self.prediction_history)):
      auc = roc_auc_score(
        self.outcome_history[i-window:i],
        self.prediction_history[i-window:i]
      )
      rolling_aucs.append(auc)

    # Check if recent AUC significantly below baseline
    recent_auc = np.mean(rolling_aucs[-100:])
    baseline_auc = np.mean(rolling_aucs[-1000:-100])

    if recent_auc < baseline_auc - 0.05:  # 5pp drop
      alert(f"Performance degradation! Recent AUC: {recent_auc:.3f}, Baseline: {baseline_auc:.3f}")
      # Trigger model retraining or reduced capital allocation

# In production
monitor = ModelMonitor()

for data in live_stream:
  features = extract_features(data)
  prediction = model.predict(features)

  # Log everything
  monitor.log_prediction(features, prediction)

  # Periodic checks (every 100 predictions)
  if len(monitor.prediction_history) % 100 == 0:
    monitor.check_feature_drift()
    monitor.check_prediction_calibration()
    monitor.check_performance_degradation()

In [ ]:
# Spend time on this
def validate_data_quality(df):
  """Check for common data issues"""
  issues = []

  # Check for duplicates
  if df.index.duplicated().any():
    issues.append("Duplicate timestamps found")

  # Check for gaps
  expected_freq = pd.infer_freq(df.index)
  if expected_freq:
    full_range = pd.date_range(df.index[0], df.index[-1], freq=expected_freq)
    missing = full_range.difference(df.index)
    if len(missing) > 0:
      issues.append(f"{len(missing)} missing timestamps")

  # Check for outliers (likely errors)
  returns = df['close'].pct_change()
  extreme_returns = returns[abs(returns) > 0.5]  # >50% moves likely errors
  if len(extreme_returns) > 0:
    issues.append(f"{len(extreme_returns)} extreme returns (possible errors)")

  # Check for stuck prices
  stuck = (df['close'].diff() == 0).rolling(10).sum() == 10
  if stuck.any():
    issues.append(f"Stuck prices detected (exchange issues?)")

  if issues:
    print("Data quality issues:")
    for issue in issues:
      print(f"  - {issue}")
    return False
  else:
    print("Data quality: OK")
    return True

# Always run this before training
validate_data_quality(training_data)